In [ ]:
# Importing libraries and defining paths

from IPython.display import display
from PIL import Image

from object_detection.config.loader import load_config
from object_detection.detection.inference import Detector
from object_detection.detection.train import train
from object_detection.utils.visualization import draw_detections

cfg = load_config()
print(cfg.data.detector_yaml)

In [ ]:
# Training the baseline model

results = train(cfg)

In [ ]:
# Loading the best weights

detector = Detector(cfg.inference.weights, cfg.inference.conf)

In [ ]:
# Inspecting one prediction

VALID = cfg.data.detector_yaml.parent / "valid"
val_imgs = sorted((VALID / "images").glob("*"))

with Image.open(val_imgs[0]) as image:
    print("image (height, width):", (image.height, image.width))

for detection in detector.predict(val_imgs[0]):
    print([round(v) for v in detection.bbox], round(detection.confidence, 3))

In [ ]:
# Visualising predictions

for img_path in val_imgs[:15]:
    detections = detector.predict(img_path)
    print(img_path.name, "-", len(detections), "detections")

    with Image.open(img_path) as image:
        annotated = draw_detections(image, detections)
    annotated.thumbnail((800, 800))
    display(annotated)

First time running the model, some images have more boxes than there are objects. This may have multiple factors, including but not limited to diverse object selection (with some having differing shapes depending on state), not enough training data, and many more.

In [ ]:
# Background check: images with no objects should get no boxes

for img_path in val_imgs:
    label_path = VALID / "labels" / (img_path.stem + ".txt")
    is_background = not label_path.exists() or label_path.read_text().strip() == ""

    if is_background:
        detections = detector.predict(img_path)
        confidences = [round(d.confidence, 3) for d in detections]
        print(img_path.name, "detections:", len(detections), confidences)

The background check shows that one of the background images have 2 predicted boxes. Since all background-only images are verified to contain no objects, this is a genuine false positive. There are a few strategies to mitigate this, including but not limited to having more training data, or raising ```conf``` to ~0.62, but this will also make the model not detect some of the lower-confidence objects which will raise the amount of false negatives. This is also one trade-off to consider and measure.

Looking at the actual image and the predictions, both predictions are for the handles of the chair which is one of the backgrounds. Perhaps it is viable to include more images of this particular background with the handles visible to maybe reduce these types of errors.